# UTS Data Acquisition & Management — Kecamatan Ubud, Gianyar

**Topik:** Akuisisi dan analisis data akomodasi pariwisata di Kecamatan Ubud, Kabupaten Gianyar, Bali  
**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Mata kuliah:** Data Acquisition & Management

> **Status pengerjaan:** kerangka notebook ini sudah disiapkan. Dataset utama akomodasi minimal 100 baris dan URL halaman scraping yang benar-benar diizinkan masih harus dimasukkan setelah diperoleh dari sumber resmi. Notebook sengaja menghentikan proses jika data wajib belum tersedia; tidak ada data contoh yang disamarkan sebagai data asli.

## Sumber resmi yang sudah ditemukan
1. BPS, *Kecamatan Ubud Dalam Angka 2025*: https://gianyarkab.bps.go.id/id/publication/2025/09/26/d7aaf2f580d8d7aa8a34ed93/kecamatan-ubud-dalam-angka-2025.html
2. BPS, *Kecamatan Ubud Dalam Angka 2024* (tabel 6.1.1 jumlah sarana akomodasi menurut desa/kelurahan dan jenis akomodasi, tahun 2023): https://web-api.bps.go.id/download.php?f=wMvs2PWxklNiwbwG5X6GnmdnWE10UXZjeHFGOFJpdFRhZ3gvNHlOemw1d0dZeG5FSDVQM1RzVGFsZEVSU1BGZDVnV2REcko4dUVhRC9lUzNyczdJVjd6WTQ3Q25KUnBmR3IvY0FGUXlEKzVlak1Bb3F5YXZhMlN4SkVBZ0QyZmFOdS9wUVNHTEFGMVRjY1NwdVZlN1VPWlVQN3pqcVViQ2o1WURuaW1GRGJVcDNzRnhQUG1ESi9CaWRXWjhTSktyQ2J3Q0tiOTN1YS9JYTBvYXRFSmZDeVZrTXJ4VjZyd3YzbDlBK0Z2RGZHQ1R4MVU4N0RiTzNjNC8yc2JpZGhQR21HNTN5U0g%3D
3. Open-Meteo Historical Weather API: https://open-meteo.com/en/docs/historical-weather-api

**Penting:** tabel BPS Ubud yang berhasil ditemukan merupakan tabel agregat desa (8 baris), bukan dataset daftar usaha akomodasi 100+ baris. Tabel itu boleh dipakai sebagai bukti pendukung, tetapi **tidak memenuhi syarat dataset utama minimal 100 baris**. Data utama harus diminta ke Dinas Pariwisata/PPID atau ditemukan dalam portal data resmi.

## 0. Persiapan dan ketentuan request

Sebelum menjalankan notebook:
- Isi email kampus pada variabel `EMAIL_KAMPUS`.
- Unggah file asli direktori akomodasi ke `data/raw/akomodasi_ubud.xlsx` (atau ubah nama/path di konfigurasi).
- Jangan mengubah file mentah.
- Pastikan setiap halaman scraping sudah diperiksa `robots.txt` dan ketentuan situs.

In [ ]:
# Mengimpor pustaka dan menyiapkan folder kerja.
import os, re, json, time, hashlib, mimetypes
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = Path(".")
RAW = BASE / "data" / "raw"
INTERIM = BASE / "data" / "interim"
PROCESSED = BASE / "data" / "processed"
META = BASE / "metadata"
for folder in [RAW, INTERIM, PROCESSED, META]:
    folder.mkdir(parents=True, exist_ok=True)

# Ganti dengan email kampus aktif sebelum mengirim request.
EMAIL_KAMPUS = "ISI_EMAIL_KAMPUS"
if "ISI_" in EMAIL_KAMPUS or "@" not in EMAIL_KAMPUS:
    raise ValueError("Isi EMAIL_KAMPUS dengan email kampus aktif sebelum menjalankan request.")

USER_AGENT = f"INSTIKI-DAM-UTS/1.0 ({EMAIL_KAMPUS})"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "application/json,text/html;q=0.9"}
TIMEOUT = 20
JEDA = 1.5
TANGGAL_AKSES = datetime.now().astimezone().isoformat(timespec="seconds")

PROV_PATH = META / "provenance.csv"

In [ ]:
# Fungsi bantu: hitung MD5 file mentah dan catat asal setiap sumber.
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(nama_sumber, url, lisensi, path_mentah, catatan=""):
    baris = {
        "sumber": nama_sumber,
        "url": url,
        "tanggal_akses": TANGGAL_AKSES,
        "lisensi_ketentuan": lisensi,
        "file_mentah": str(path_mentah),
        "md5": md5_file(path_mentah) if Path(path_mentah).exists() else "",
        "catatan": catatan
    }
    df = pd.DataFrame([baris])
    if PROV_PATH.exists():
        df = pd.concat([pd.read_csv(PROV_PATH), df], ignore_index=True)
        df = df.drop_duplicates(subset=["sumber", "file_mentah"], keep="last")
    df.to_csv(PROV_PATH, index=False)
    return df.tail(1)

def get_with_retry(url, params=None, max_retry=3):
    # Semua request memakai timeout, User-Agent, jeda, dan retry untuk gangguan sementara.
    for attempt in range(1, max_retry + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            print(f"Percobaan {attempt}: HTTP {response.status_code} — {response.url}")
            if response.status_code == 200:
                time.sleep(JEDA)
                return response
            if response.status_code in [429, 500, 502, 503, 504] and attempt < max_retry:
                time.sleep(JEDA * attempt)
                continue
            response.raise_for_status()
        except requests.RequestException:
            if attempt == max_retry:
                raise
            time.sleep(JEDA * attempt)
    raise RuntimeError("Request gagal setelah seluruh percobaan.")

# Soal 1 — Membaca dan mengidentifikasi dataset utama

Dataset utama yang dibutuhkan adalah **direktori usaha akomodasi di Kecamatan Ubud**, dengan satu baris mewakili satu usaha akomodasi. Kolom yang diharapkan misalnya nama usaha, desa/kelurahan, jenis akomodasi, jumlah kamar, dan status operasional (jika tersedia).

**Jangan menganggap tabel ringkasan BPS 8 desa sebagai dataset utama**, karena jumlah barisnya kurang dari 100.

In [ ]:
# Baca file dataset utama. Simpan file asli terlebih dahulu di folder data/raw.
FILE_DATASET = RAW / "akomodasi_ubud.xlsx"

if not FILE_DATASET.exists():
    raise FileNotFoundError(
        "File utama belum tersedia. Minta/unduh dataset resmi akomodasi Ubud "
        "minimal 100 baris, lalu simpan sebagai data/raw/akomodasi_ubud.xlsx."
    )

# Membaca Excel. Jika sumber berupa CSV, gunakan pd.read_csv dengan sep dan encoding yang sesuai.
df = pd.read_excel(FILE_DATASET, sheet_name=0)

# Pemeriksaan awal jumlah baris dan kolom.
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Nama kolom:", df.columns.tolist())
if len(df) < 100:
    raise ValueError(f"Dataset hanya {len(df)} baris; syarat UTS minimal 100 baris belum terpenuhi.")

display(df.head())

In [ ]:
# Mengidentifikasi tipe data, nilai kosong, dan ukuran file.
info_tipe = pd.DataFrame({
    "kolom": df.columns,
    "tipe_data": [str(t) for t in df.dtypes],
    "jumlah_kosong": [int(df[c].isna().sum()) for c in df.columns],
    "jumlah_unik": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
ukuran_byte = FILE_DATASET.stat().st_size
ukuran_mb = ukuran_byte / (1024 ** 2)

print("Jenis data: terstruktur (tabel Excel dengan baris dan kolom).")
print(f"Ukuran file: {ukuran_byte:,} byte = {ukuran_mb:.4f} MB")
print("Estimasi storage 1 tahun = ukuran file x 12 x 1,30")
storage_tahun_mb = ukuran_mb * 12 * 1.30
print(f"Estimasi storage: {storage_tahun_mb:.4f} MB")

display(info_tipe)

### Interpretasi Soal 1 (lengkapi setelah dataset terbaca)

- **Jenis data:** terstruktur, karena disajikan dalam baris dan kolom.
- **Jumlah baris/kolom:** diambil dari output notebook, bukan dikarang.
- **Tipe data:** lihat tabel `info_tipe`.
- **Satuan:** jelaskan berdasarkan definisi setiap kolom; misalnya jumlah kamar dalam unit kamar, jika kolom tersebut tersedia.
- **Cakupan wilayah:** Kecamatan Ubud, Kabupaten Gianyar, Bali.
- **Cakupan waktu:** ikuti tahun/periode yang tertulis pada metadata dataset sumber.
- **Ukuran berkas:** hasil `stat()` pada file mentah.
- **Asumsi storage:** diasumsikan ukuran data bulanan sama dengan ukuran file yang dianalisis; 12 bulan ditambah cadangan 30%.

# Soal 2 — Lisensi, ketentuan portal, PII, quasi-identifier, dan Data Ethics Checklist

Lisensi **harus disalin dari halaman dataset/portal yang benar-benar digunakan**. Jangan menebak lisensi. Simpan bukti berupa URL halaman metadata, tangkapan layar atau salinan halaman lisensi, dan tanggal akses.

Kolom PII yang perlu diwaspadai: nama orang, nomor telepon pribadi, email pribadi, NIK, dan identitas personal. Quasi-identifier dapat berupa kombinasi lokasi rinci, jabatan, atau atribut lain yang memungkinkan identifikasi seseorang. Nama usaha dan alamat publik usaha tidak otomatis merupakan PII pribadi, tetapi tetap perlu dinilai konteksnya.

In [ ]:
# Buat checklist etika awal. Ubah status dan bukti sesuai pemeriksaan nyata terhadap dataset dan portal.
checklist = pd.DataFrame([
    [1, "Lisensi dataset telah diperiksa", "Perlu verifikasi", "Tempel URL/halaman lisensi resmi"],
    [2, "Ketentuan penggunaan portal telah dibaca", "Perlu verifikasi", "Tempel URL syarat penggunaan portal"],
    [3, "Sumber dan pemilik data dicantumkan", "Ya jika metadata tersedia", "Isi nama instansi dan URL sumber"],
    [4, "Tujuan penggunaan data jelas dan akademik", "Ya", "UTS Data Acquisition & Management"],
    [5, "Kolom PII diperiksa", "Perlu pemeriksaan kolom", "Daftar kolom dan hasil pemeriksaan"],
    [6, "Quasi-identifier diperiksa", "Perlu pemeriksaan kolom", "Daftar kolom lokasi/atribut kombinatif"],
    [7, "Data sensitif diminimalkan/dihindari", "Perlu verifikasi", "Pastikan NIK, kontak pribadi, dll. tidak disebarkan"],
    [8, "Kualitas dan keterbatasan data diperiksa", "Perlu verifikasi", "Cek nilai kosong, duplikasi, rentang nilai"],
    [9, "Atribusi dan provenance dicatat", "Perlu verifikasi", "URL, tanggal akses, lisensi, MD5"],
], columns=["no", "pemeriksaan", "status_awal", "bukti_yang_harus_dilengkapi"])

checklist.to_excel(META / "data_ethics_checklist.xlsx", index=False)
display(checklist)

# Soal 3 — Akuisisi data melalui API

API yang dipakai sebagai data pendukung adalah **Open-Meteo Historical Weather API** untuk koordinat Ubud. Ini bukan data jumlah akomodasi; fungsinya sebagai konteks cuaca wilayah. Data harian diambil selama 31 hari agar melebihi syarat minimal 30 baris.

Dokumentasikan lisensi dan ketentuan Open-Meteo dari halaman resminya pada tanggal akses. Open-Meteo menyediakan data cuaca historis dan API publik; periksa ketentuan atribusi yang berlaku sebelum pengumpulan.

In [ ]:
# Mengambil data cuaca historis harian untuk Ubud, Bali.
API_URL = "https://archive-api.open-meteo.com/v1/archive"
LAT_UBUD, LON_UBUD = -8.5069, 115.2625
PARAMS = {
    "latitude": LAT_UBUD,
    "longitude": LON_UBUD,
    "start_date": "2025-10-01",
    "end_date": "2025-10-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar"
}

r_api = get_with_retry(API_URL, params=PARAMS)
api_raw = RAW / "open_meteo_ubud_2025-10.json"
api_raw.write_bytes(r_api.content)  # simpan respons mentah tanpa diubah
payload = r_api.json()

daily = payload["daily"]
cuaca = pd.DataFrame({
    "tanggal": pd.to_datetime(daily["time"]),
    "suhu_maks_c": daily["temperature_2m_max"],
    "suhu_min_c": daily["temperature_2m_min"],
    "curah_hujan_mm": daily["precipitation_sum"],
    "angin_maks_kmh": daily["wind_speed_10m_max"]
})
cuaca["wilayah"] = "Kecamatan Ubud"
assert len(cuaca) >= 30, "Data API kurang dari 30 baris."
assert not cuaca["tanggal"].duplicated().any(), "Ada tanggal duplikat."
cuaca.to_csv(INTERIM / "cuaca_ubud.csv", index=False)
catat_provenance(
    "Open-Meteo Historical Weather API", r_api.url,
    "Periksa lisensi/atribusi resmi Open-Meteo pada tanggal akses",
    api_raw, "Data cuaca harian untuk Ubud; bukan data akomodasi."
)
display(cuaca.head())
print("Jumlah baris API:", len(cuaca))

# Soal 4 — Pemeriksaan robots.txt dan web scraping

Bagian ini **tidak boleh diarahkan ke situs terlarang** yang disebut dalam soal (Booking.com, Agoda, Traveloka, tiket.com, Airbnb, Tripadvisor, Google Maps, dan media sosial).

Pilih halaman statis resmi yang relevan, misalnya halaman tabel/daftar dari BPS atau portal pemerintah. Masukkan URL halaman yang benar-benar berisi tabel/listing yang akan diambil. Notebook memeriksa `robots.txt` terlebih dahulu dan hanya melanjutkan jika diizinkan.

**Jangan mengisi URL asal-asalan.** Halaman target dan lisensinya harus diperiksa dan dicatat.

In [ ]:
# Isi URL halaman statis resmi yang relevan setelah menemukan halaman tabel/listing.
URL_TARGET = ""  # contoh: URL halaman tabel resmi, bukan URL hasil pencarian

if not URL_TARGET:
    raise ValueError(
        "URL_TARGET belum diisi. Pilih halaman statis resmi yang relevan, "
        "periksa ketentuan situs, lalu isi URL sebelum menjalankan scraping."
    )

parsed = urlparse(URL_TARGET)
URL_ROBOTS = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

# Unduh robots.txt terlebih dahulu dengan timeout dan User-Agent.
r_robots = get_with_retry(URL_ROBOTS)
robots_raw = RAW / "robots_target.txt"
robots_raw.write_bytes(r_robots.content)

rp = RobotFileParser()
rp.parse(r_robots.text.splitlines())
boleh_scrape = rp.can_fetch(USER_AGENT, URL_TARGET)
crawl_delay = rp.crawl_delay(USER_AGENT)
print("URL robots:", URL_ROBOTS)
print("URL target:", URL_TARGET)
print("Diizinkan robots.txt:", boleh_scrape)
print("Crawl-delay:", crawl_delay)

if not boleh_scrape:
    raise PermissionError("robots.txt tidak mengizinkan pengambilan halaman target.")

catat_provenance("robots.txt halaman target", URL_ROBOTS,
                 "Robots exclusion protocol; bukan lisensi konten",
                 robots_raw, f"Diizinkan={boleh_scrape}; crawl-delay={crawl_delay}")

In [ ]:
# Ambil halaman target hanya setelah robots.txt menyatakan boleh.
time.sleep(max(JEDA, crawl_delay or 0))
r_page = get_with_retry(URL_TARGET)
html_raw = RAW / "halaman_target_scraping.html"
html_raw.write_bytes(r_page.content)  # simpan HTML mentah tanpa perubahan

soup = BeautifulSoup(r_page.text, "html.parser")
tables = soup.find_all("table")
print("Jumlah tabel HTML ditemukan:", len(tables))

if not tables:
    raise ValueError("Tidak ditemukan tabel HTML. Pilih halaman statis yang memiliki tabel atau listing.")

# Tampilkan ringkasan semua tabel agar tabel relevan dipilih secara sadar.
for i, table in enumerate(tables):
    print(f"Tabel {i}: {table.get_text(' ', strip=True)[:250]}")

# Setelah melihat ringkasan, ubah indeks jika tabel relevan bukan tabel pertama.
INDEX_TABEL = 0
hasil_tabel = pd.read_html(str(tables[INDEX_TABEL]))[0]
hasil_tabel.to_csv(INTERIM / "hasil_scraping.csv", index=False)

catat_provenance("Halaman statis hasil scraping", URL_TARGET,
                 "Isi lisensi/ketentuan halaman resmi yang diperiksa",
                 html_raw, "Scraping dilakukan setelah pemeriksaan robots.txt.")
display(hasil_tabel.head())

# Soal 5 — Penggabungan minimal dua sumber

Contoh penggabungan yang paling aman untuk notebook ini adalah data akomodasi dengan ringkasan cuaca Ubud. Dataset akomodasi harus memiliki kolom nama wilayah yang dapat dinormalisasi menjadi `wilayah`; karena cakupan dataset adalah satu kecamatan, nilai wilayah diseragamkan menjadi `Kecamatan Ubud`. Ringkasan cuaca dibuat satu baris agar validasi `many_to_one` dapat digunakan.

Penggabungan ini menambahkan konteks cuaca rata-rata periode API pada setiap baris usaha akomodasi. Jelaskan bahwa ini **bukan** cuaca spesifik pada tanggal transaksi atau kondisi operasional setiap usaha.

In [ ]:
# Normalisasi data utama dan siapkan satu baris ringkasan cuaca Ubud.
akomodasi = df.copy()
akomodasi.columns = [str(c).strip().lower().replace(" ", "_") for c in akomodasi.columns]
akomodasi["wilayah"] = "Kecamatan Ubud"

ringkasan_cuaca = pd.DataFrame([{
    "wilayah": "Kecamatan Ubud",
    "periode_cuaca_mulai": cuaca["tanggal"].min().date().isoformat(),
    "periode_cuaca_selesai": cuaca["tanggal"].max().date().isoformat(),
    "rata_suhu_maks_c": cuaca["suhu_maks_c"].mean(),
    "rata_suhu_min_c": cuaca["suhu_min_c"].mean(),
    "total_curah_hujan_mm": cuaca["curah_hujan_mm"].sum(),
    "rata_angin_maks_kmh": cuaca["angin_maks_kmh"].mean()
}])

gabungan = akomodasi.merge(
    ringkasan_cuaca,
    on="wilayah",
    how="left",
    validate="many_to_one"
)

# Bukti kualitas: jumlah baris tidak bertambah dan tidak ada duplikasi seluruh baris.
assert len(gabungan) == len(akomodasi), "Jumlah baris berubah setelah merge."
duplikat = int(gabungan.duplicated().sum())
print("Baris sebelum merge:", len(akomodasi))
print("Baris sesudah merge:", len(gabungan))
print("Duplikasi seluruh baris:", duplikat)

gabungan.to_csv(PROCESSED / "data_gabungan_ubud.csv", index=False)
display(gabungan.head())

In [ ]:
# Simpan metadata ringkas hasil penggabungan.
metadata_hasil = pd.DataFrame([{
    "file_hasil": str(PROCESSED / "data_gabungan_ubud.csv"),
    "jumlah_baris": len(gabungan),
    "jumlah_kolom": len(gabungan.columns),
    "duplikasi_baris": int(gabungan.duplicated().sum()),
    "metode_merge": "left",
    "validate": "many_to_one",
    "sumber_1": "Dataset resmi direktori akomodasi Kecamatan Ubud (wajib dilengkapi)",
    "sumber_2": "Open-Meteo Historical Weather API",
    "tanggal_proses": datetime.now().astimezone().isoformat(timespec="seconds")
}])
metadata_hasil.to_csv(META / "metadata_hasil_penggabungan.csv", index=False)

print("Provenance:")
display(pd.read_csv(PROV_PATH) if PROV_PATH.exists() else pd.DataFrame())
print("Metadata hasil:")
display(metadata_hasil)

## Kesimpulan

Notebook ini menghasilkan alur pembacaan dataset, pemeriksaan etika, akuisisi API, pemeriksaan robots.txt dan scraping, serta penggabungan data. **Tugas belum boleh dinyatakan selesai** sampai:
1. Dataset utama resmi akomodasi Ubud minimal 100 baris tersedia dan lolos pemeriksaan.
2. Lisensi dan ketentuan portal dataset utama dibuktikan.
3. URL scraping resmi yang relevan telah diisi, diizinkan robots.txt, dan hasil tabelnya telah diperiksa.
4. Semua sel dijalankan ulang dari awal tanpa error.
5. File mentah, hasil, checklist, dan provenance disertakan saat pengumpulan.